# Lecture 3.2 — Shell Tools: Bash & Monitor
### Build Production AI Agents with the Claude Agent SDK
**Section 3 — Built-In Tools Deep Dive**

---

In this notebook we explore the two **shell tools** in the Claude Agent SDK:

| Tool | What it does |
|------|--------------|
| **Bash** | Runs any terminal command — system checks, scripts, git operations |
| **Monitor** | Runs a script in the background and feeds each stdout line to Claude as it appears |

This notebook is **self-contained** — its own setup cells, its own sample files, no dependency on any previous lecture.

**What we cover:**
- **Part 1** — The Bash Tool: what it does, safety note, demo
- **Part 2** — The Monitor Tool: architecture, async task lifecycle, full message surfacing
- **Part 3** — Bash vs Monitor: the mental model

In [1]:
# Cell 2 — Install the SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 10.5 MB/s eta 0:00:00


In [2]:
# Cell 3 — Load API key from Colab Secrets
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

The cell below sets the Claude model used throughout this notebook.
Change `MODEL_NAME` to any model you have access to.

For the latest available models:
[https://platform.claude.com/docs/en/about-claude/models/overview](https://platform.claude.com/docs/en/about-claude/models/overview)

In [3]:
# Cell 5 — Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

---
## Part 1 — The Bash Tool

### What it does

The **Bash** tool gives your agent the ability to run any shell command on the underlying system:
- Terminal commands (`python --version`, `pwd`, `df -h`, `date`)
- Build scripts and shell scripts
- Git operations (`git status`, `git log`)
- Package installs and system checks

The agent decides **which commands to run** based on your goal. You describe what you want in plain English — Claude decides how to get it.

### When to use it

Use Bash whenever your agent needs to interact with the **operating system** — system info, scripts, git, anything a developer would do in a terminal.

### ⚠️ Safety Note

> **Bash is the most powerful tool in the Claude Agent SDK.** It can run any command on the system — including destructive ones.
>
> Be deliberate about every task you give an agent with Bash access.
>
> **Section 4 — Permissions & Safety** covers how to control what your agent is and is not allowed to do.

In [4]:
# Cell 7 — Bash tool demo: system information report
from claude_agent_sdk import query, ClaudeAgentOptions

async for message in query(
    prompt="""
    Use bash commands to find out the following system information:
    1. What version of Python is installed?
    2. What is the current working directory?
    3. How much disk space is available?
    4. What is the current date and time?
    Present the results as a clean, readable system report.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Bash"],
        model=MODEL_NAME,
    ),
):
    if hasattr(message, "result"):
        print(message.result)

## System Information Report

Here's your clean, readable system report:

---

### **Python Version**
- **Python 3.12.13** ✓

### **Current Working Directory**
- `/content`

### **Disk Space**
- **Filesystem:** overlay
- **Total Size:** 108G
- **Used:** 21G
- **Available:** 88G
- **Usage:** 20% full ✓

### **Date & Time**
- **Saturday, June 6, 2026**
- **Time:** 3:40:39 PM UTC

---

**Summary:** Your system is healthy with Python 3.12.13 installed, running from the `/content` directory, and has plenty of disk space available (80% free).


---
## Part 2 — The Monitor Tool

### What it does

The **Monitor** tool runs a script in the background and feeds each line of stdout to Claude **as it appears** — Claude can react before the process finishes.

### Bash vs Monitor

| | Bash | Monitor |
|---|---|---|
| Execution | Foreground | Background |
| Output delivery | All at once when done | Line by line as it appears |
| Best for | Quick commands (seconds) | Long-running processes (minutes) |
| Real-time reaction | No | Yes |

### How Monitor actually works — the async architecture

When Claude calls Monitor, this is what happens:

1. Claude issues a `ToolUseBlock` — calling Monitor with a command
2. A **single** `ToolResultBlock` comes back immediately: *"Monitor started, task id X, you will be notified on each event"*
3. A `TaskStartedMessage` fires — the background task is now live. Its `task_type` field reads `local_bash`, revealing that **Bash is Monitor's silent execution engine**
4. As the subprocess produces each stdout line, the Claude Code CLI **injects that line as a fresh `UserMessage`** directly into Claude's conversation context — a real API call, real tokens billed
5. Claude responds to each injected line with a new `AssistantMessage` — your Python loop sees these arriving progressively
6. A `TaskNotificationMessage` fires when the subprocess finishes

### What the SDK surfaces vs hides

Those injected `UserMessage`s are **never surfaced to your Python loop** — just like your initial prompt is not surfaced. The SDK gives you outputs, not the inputs it manages on your behalf.

The one exception: `ToolResultBlock`s inside `UserMessage`s **are** surfaced — because they close a tool call Claude explicitly made.

| Message | Surfaced to Python loop? |
|---|---|
| Your initial prompt (`UserMessage`) | No — you already know it |
| `ToolResultBlock` inside `UserMessage` | **Yes** — closes a tool call Claude made |
| Injected stdout line (`UserMessage`) | No — CLI internal, autonomous |
| `AssistantMessage` (Claude's reactions) | **Yes** — this is the output |

### Bash must be in `allowed_tools`

Monitor has no shell of its own. It needs Bash as its execution engine to launch the subprocess. Bash never appears as an explicit `ToolUseBlock` — but without it, Claude cannot launch the script directly. Instead it reads the file first, predicts the output, and only then runs it — meaning its reactions are based on prediction, not genuine real-time observation.

Always: `allowed_tools=["Monitor", "Bash"]`

In [5]:
# Cell 9 — Create the build script
#
# flush=True is critical.
# Without it, Python buffers stdout and the OS delivers all lines to Monitor
# in one batch when the process exits — defeating the purpose entirely.
#
# print(step, flush=True) is sufficient on its own.
# sys.stdout.flush() after it would be redundant — they do the same thing.

with open("/content/build_script.py", "w") as f:
    f.write("import time\n\n")
    f.write("steps = [\n")
    f.write("    'Build started...',\n")
    f.write("    'Compiling modules...',\n")
    f.write("    'Running tests...',\n")
    f.write("    'Packaging application...',\n")
    f.write("    'Build complete! All steps passed.',\n")
    f.write("]\n\n")
    f.write("for step in steps:\n")
    f.write("    print(step, flush=True)\n")
    f.write("    time.sleep(3)\n")

### Verifying the build script

Let us print the script so we can see exactly what the agent will be watching.
Each `print(step, flush=True)` will arrive at Monitor as a **separate event** —
delivered immediately, not buffered.

In [6]:
# Cell 11 — Print build script contents
print("Build script contents:")
print("=" * 40)
with open("/content/build_script.py", "r") as f:
    print(f.read())

Build script contents:
import time

steps = [
    'Build started...',
    'Compiling modules...',
    'Running tests...',
    'Packaging application...',
    'Build complete! All steps passed.',
]

for step in steps:
    print(step, flush=True)
    time.sleep(3)



### Monitor demo — surfacing the full message stream

The cell below surfaces **every message type** in a Monitor run with clear labels:

| Prefix | What it shows |
|--------|---------------|
| 💭 Thinking | Claude's internal reasoning (`ThinkingBlock`) |
| 🔧 Launching | Claude calling the Monitor tool (`ToolUseBlock`) |
| 📤 Tool result | System confirming Monitor started (`ToolResultBlock` in `UserMessage`) |
| ⚙️ Task started | Background task live — note `task_type: local_bash` (`TaskStartedMessage`) |
| 🤖 Claude | Claude reacting to each build line (`TextBlock` in `AssistantMessage`) |
| 📨 Task done | Subprocess finished (`TaskNotificationMessage`) |
| Final Result | Last `ResultMessage` only — via `last_result` pattern |

> ⏱ **This cell takes ~15–18 seconds** — five steps × 3 seconds + API round trips.
> Watch the 🤖 Claude reactions arrive progressively with ~3 second gaps between them.
> That is Monitor delivering one event per flushed stdout line.

In [7]:
# Cell 13 — Monitor demo with full message stream surfaced
from claude_agent_sdk import (
    query, ClaudeAgentOptions,
    AssistantMessage, ResultMessage, UserMessage
)
from claude_agent_sdk.types import (
    TextBlock, ToolUseBlock, ThinkingBlock, ToolResultBlock,
    TaskStartedMessage, TaskNotificationMessage
)

# last_result pattern:
# ResultMessage fires once per turn, not once total.
# Keep overwriting — last one wins. Print once after the loop.
last_result = None

async for message in query(
    prompt="""
    Monitor the script at /content/build_script.py using Python.
    After EACH individual line of output appears, immediately print
    a one-sentence reaction to that specific line before waiting for
    the next one. Do not wait for the script to finish.
    When the script ends, provide a final summary.
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Monitor", "Bash"],
        model=MODEL_NAME,
    ),
):
    # AssistantMessage — everything Claude says and decides
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, ThinkingBlock):
                print(f"💭 Thinking: {block.thinking.strip()}\n")

            elif isinstance(block, ToolUseBlock) and block.name == "Monitor":
                print(f"🔧 Launching monitor: {block.input.get('command')}")

            elif isinstance(block, TextBlock) and block.text.strip():
                print(f"🤖 Claude: {block.text.strip()}\n")

    # UserMessage — ToolResultBlocks ARE surfaced (they close a tool call Claude made)
    # Injected stdout UserMessages are NOT surfaced by the SDK
    elif isinstance(message, UserMessage):
        for block in message.content:
            if isinstance(block, ToolResultBlock):
                print(f"📤 Tool result: {block.content}\n")

    # TaskStartedMessage — background task is live
    # task_type: local_bash confirms Bash is Monitor's silent execution engine
    elif isinstance(message, TaskStartedMessage):
        print(f"⚙️  Task started — id: {message.task_id}, type: {message.task_type}\n")

    # TaskNotificationMessage — subprocess finished
    elif isinstance(message, TaskNotificationMessage):
        print(f"📨 Task done — {message.summary}\n")

    # ResultMessage — one per turn, keep overwriting, last one wins
    elif isinstance(message, ResultMessage) and message.result:
        last_result = message.result

print("--- Final Result ---")
print(last_result)

💭 Thinking: The user wants me to monitor a Python script at `/content/build_script.py` and:
1. Execute it
2. After each line of output, immediately print a one-sentence reaction to that specific line
3. Don't wait for the script to finish - be reactive to each line
4. Provide a final summary when the script ends

This is a streaming/monitoring task. I should use the Monitor tool to stream output from the script. Let me check if Monitor is available - yes, it's in the deferred tools list. I need to fetch its schema first using ToolSearch.

Actually, looking at the tools available, I see Monitor is listed in the deferred tools. Let me first fetch its schema, then use it to monitor the Python script execution.

📤 Tool result: [{'type': 'tool_reference', 'tool_name': 'Monitor'}]

💭 Thinking: The user wants me to monitor the script at /content/build_script.py using Python. They want me to:
1. Monitor the output line by line
2. After EACH line appears, print a one-sentence reaction immediate

---
## Part 3 — Bash vs Monitor: When to Use Which

### The mental model

| Use | When |
|-----|------|
| **Bash** | Command finishes quickly. Full output at once is fine. System checks, git, short scripts. |
| **Monitor** | Process runs for a long time. You want the agent to react as output arrives. Builds, test suites, log watchers. |

### Rule of thumb

> - Finishes in **seconds** → **Bash**
> - Runs for **minutes**, want real-time reactions → **Monitor**

### Checklist for Monitor

- [ ] `flush=True` on every `print()` in the script — without it, buffered stdout defeats the purpose
- [ ] `allowed_tools=["Monitor", "Bash"]` — Bash is Monitor's silent execution engine
- [ ] Use `last_result` pattern — `ResultMessage` fires once per turn, not once total
- [ ] Surface `UserMessage` → `ToolResultBlock` separately from `AssistantMessage` content
- [ ] Remember: injected stdout `UserMessage`s are real API calls with real token cost — they just never appear in your Python loop

---
## Summary

In this lecture we covered:

- **Bash** — runs any terminal command; Claude chose `python --version`, `pwd`, `df -h`, `date` from a plain English goal
- **Monitor** — runs a script in the background; each flushed stdout line is injected by the CLI as a real `UserMessage` API call driving a new Claude reaction
- **flush=True** — `print(step, flush=True)` is sufficient; `sys.stdout.flush()` after it is redundant
- **Bash as silent engine** — `task_type: local_bash` in `TaskStartedMessage` confirms it; without Bash, Claude predicts rather than monitors
- **SDK surfaces outputs not inputs** — injected `UserMessage`s are hidden; `ToolResultBlock` `UserMessage`s are the one exception
- **last_result pattern** — `ResultMessage` fires per turn; overwrite and print once after the loop

---

### Up next — Lecture 3.3: Search Tools: Glob & Grep

**Glob** finds files by pattern (`**/*.py`, `src/**/*.ts`).
**Grep** searches inside file contents using regex.

If Bash gave your agent a terminal — Glob and Grep give it a search engine.